# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
# Imported the userdata module from google.colab
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

print("Hugging Face token successfully loaded (masked for security).")

Hugging Face token successfully loaded (masked for security).


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis

One row in this dataset represents a **unique search query** performed by a **specific user** on a **given day**. This means that if a user performs the same search query multiple times on the same day, it should ideally be aggregated into a single row, or specific identifying features (like `query_id` or `session_id`) should distinguish them if the task requires finer granularity.

### Time Window

The dataset covers a specific range of dates. The minimum and maximum dates present in the data are verified to understand the full time window of the analysis.

In [15]:
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login
from google.colab import userdata
import itertools # Imported itertools for slicing

# Retrieved the Hugging Face token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face Hub
login(token=hf_token)

# Load the dataset in streaming mode
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train", token=hf_token)

# Load only the first 500 rows for inspection
print("Loading first 500 rows of the dataset...")
df = pd.DataFrame(list(itertools.islice(ds, 500)))

print("First 500 rows loaded successfully. Displaying head:")
# Display the first few rows to understand the data structure
print("DataFrame Head:")
display(df.head())

# Verify the proposed unit of analysis

# Convert 'date' column to datetime objects if not already
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
elif 'timestamp' in df.columns:
    df['date'] = pd.to_datetime(df['timestamp'], errors='coerce').dt.date
    print("Using 'timestamp' column and extracting date.")
else:
    print("Warning: 'date' or 'timestamp' column not found. Cannot verify time window or daily unit of analysis effectively.")


# Check for duplicates based on the assumed unit of analysis
if 'user_id' in df.columns and 'query' in df.columns and 'date' in df.columns:
    unique_rows = df.drop_duplicates(subset=['user_id', 'query', 'date'])
    if len(unique_rows) == len(df):
        print("\nVerification: Each row appears to be a unique combination of user, query, and date.")
    else:
        print(f"\nWarning: Found {len(df) - len(unique_rows)} duplicate rows based on user, query, and date. Original rows: {len(df)}, Unique rows: {len(unique_rows)}.")
        print("Consider aggregating or reviewing the definition of a unique row.")
else:
    print("\nWarning: 'user_id', 'query', or 'date' column(s) not found. Cannot fully verify unit of analysis.")

# Verify the time window
if 'date' in df.columns and df['date'].notna().any():
    min_date = df['date'].min()
    max_date = df['date'].max()
    print(f"\nTime Window: From {min_date} to {max_date}")
else:
    print("\nCannot determine time window: 'date' column is missing or contains no valid dates.")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading first 500 rows of the dataset...
First 500 rows loaded successfully. Displaying head:
DataFrame Head:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0




Cannot determine time window: 'date' column is missing or contains no valid dates.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields: Feature / Label / Context / Excluded

To frame our machine learning task, it's crucial to categorize each field in the dataset based on its role. This helps in understanding data dependencies, potential leakage, and model interpretability.

*   **Features:** These are the input variables used by the model to make predictions. They describe the characteristics or attributes of the unit of analysis.
*   **Labels:** This is the target variable that the model is trying to predict or explain. It's the outcome we are interested in.
*   **Context:** These fields provide additional information that might be useful for analysis, filtering, or understanding the data, but are not directly used as features or labels in the core model. They can be crucial for debugging or post-analysis.
*   **Excluded:** These fields are explicitly removed from the dataset for various reasons, such as containing sensitive information, being redundant, causing data leakage, having too many missing values, or not being relevant to the prediction task.

In [16]:
# Get all column names from the DataFrame
all_columns = df.columns.tolist()
print(f"All columns in the DataFrame: {all_columns}\n")


# Label: The outcome we want to predict
labels = [
    'gsc_clicks'
]

# Context fields: Identifiers and dates are typically context.
# These provide useful information but are often not directly used as predictive features in their raw form.
context_fields = [
    'report_date',
    'client_hash_id',
    'content_hash_id'
]

# Excluded fields: No obvious exclusions from this list for a general search intelligence task.
# If there were sensitive or purely internal logging fields, they would go here.
excluded_fields = [
    # Example: 'internal_log_id', 'raw_ip_address'
]

# Features: All remaining columns that can be used to predict the label.
features = [
    col for col in all_columns
    if col not in labels and col not in context_fields and col not in excluded_fields
]

# Verify that all columns are accounted for in one category
# Convert to sets for easy comparison
features_set = set(features)
labels_set = set(labels)
context_fields_set = set(context_fields)
excluded_fields_set = set(excluded_fields)

# Check for overlaps
overlaps = (
    (features_set & labels_set) |
    (features_set & context_fields_set) |
    (features_set & excluded_fields_set) |
    (labels_set & context_fields_set) |
    (labels_set & excluded_fields_set) |
    (context_fields_set & excluded_fields_set)
)

if overlaps:
    print(f"\nWarning: Overlapping fields detected! {overlaps}. Each field should belong to only one category.")

# Check if any column is missed
all_categorized_fields = features_set | labels_set | context_fields_set | excluded_fields_set
uncategorized_columns = set(all_columns) - all_categorized_fields

if uncategorized_columns:
    print(f"\nWarning: The following columns are not categorized: {uncategorized_columns}. Please ensure all columns are assigned to a category.")

print(f"\nFeatures ({len(features)}): {features}")
print(f"Labels ({len(labels)}): {labels}")
print(f"Context Fields ({len(context_fields)}): {context_fields}")
print(f"Excluded Fields ({len(excluded_fields)}): {excluded_fields}")

All columns in the DataFrame: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


Features (26): ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_met

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Queries

This section aims to verify the unit of analysis, check data completeness (counts and missing values), and confirm the time window using specific data queries.

We will use `df.info()`, `df.isnull().sum()`, and specific `groupby` and `value_counts` operations to confirm our understanding of the data's structure and content.

In [17]:
# --- 3.1 Verify Unit of Analysis (Grain) ---
# The assumed unit of analysis is a unique combination of 'client_hash_id', 'content_hash_id', and 'report_date'.
# Let's count unique combinations and compare it to the total number of rows.

unit_of_analysis_cols = ['client_hash_id', 'content_hash_id', 'report_date']

# Ensure 'report_date' is datetime type for consistent comparison.
if 'report_date' in df.columns:
    df['report_date'] = pd.to_datetime(df['report_date'], errors='coerce')
else:
    print("Warning: 'report_date' column not found. Cannot verify time window or daily unit of analysis effectively.")

# Check if all columns for unit of analysis exist before proceeding
if all(col in df.columns for col in unit_of_analysis_cols):
    total_rows = len(df)
    unique_combinations = df.drop_duplicates(subset=unit_of_analysis_cols)
    num_unique_combinations = len(unique_combinations)

    print(f"\nTotal rows in DataFrame: {total_rows}")
    print(f"Number of unique {unit_of_analysis_cols} combinations: {num_unique_combinations}")

    if total_rows == num_unique_combinations:
        print("Grain verification successful: Each row appears to represent a unique unit of analysis.")
    else:
        print(f"Warning: {total_rows - num_unique_combinations} duplicate rows found based on the defined unit of analysis. Consider aggregating or re-evaluating the unit definition.")
else:
    print(f"Warning: One or more columns for unit of analysis ({unit_of_analysis_cols}) not found. Cannot verify grain effectively.")

# --- 3.2 Counts and Missing Values ---
print("\n--- DataFrame Info (Counts and Data Types) ---")
df.info()

print("\n--- Missing Values per Column ---")
missing_values = df.isnull().sum()
display(missing_values[missing_values > 0].sort_values(ascending=False))

# --- 3.3 Verify Time Window ---
if 'report_date' in df.columns and df['report_date'].notna().any():
    min_date = df['report_date'].min()
    max_date = df['report_date'].max()
    print(f"\nVerified Time Window: From {min_date} to {max_date}")

    # Optionally, check daily counts to see if there are gaps or uneven distribution
    print("\nDaily row counts for report_date:")
    display(df['report_date'].value_counts().sort_index())
else:
    print("\nCannot verify time window: 'report_date' column is missing or contains no valid dates.")


Total rows in DataFrame: 500
Number of unique ['client_hash_id', 'content_hash_id', 'report_date'] combinations: 500
Grain verification successful: Each row appears to represent a unique unit of analysis.

--- DataFrame Info (Counts and Data Types) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   report_date               500 non-null    datetime64[ns]
 1   client_hash_id            500 non-null    object        
 2   content_hash_id           500 non-null    object        
 3   client_has_gsc            500 non-null    bool          
 4   client_has_ga4            500 non-null    bool          
 5   gsc_data_available        500 non-null    bool          
 6   ga4_data_available        500 non-null    bool          
 7   gsc_impressions           500 non-null    int64         
 8   gsc_clicks      

,0



Verified Time Window: From 2025-01-27 00:00:00 to 2025-01-28 00:00:00

Daily row counts for report_date:


,count
report_date,
2025-01-27,303
2025-01-28,197


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits: What this data can never tell you

Understanding the limitations of a dataset is as crucial as understanding its strengths. This section highlights aspects that the current data contract, despite its insights, cannot fully capture or might misrepresent due to inherent collection methodologies or scope limitations.

1.  **Unbalanced History**: The historical depth might vary significantly across different entities (e.g., users, content, queries). Some entities might have a rich, long history, while others might appear more recently or have sporadic data points. This can make it challenging to model long-term trends or user behavior for all entities consistently.
2.  **Source-Specific Biases (e.g., GSC-only early rows)**: If the data is primarily sourced from specific platforms (like Google Search Console for early rows), it might inherently carry biases from that source. GSC, for instance, focuses on organic search performance and might exclude direct traffic, paid search, or other crucial interaction types. This could lead to an incomplete picture of user engagement or content performance, especially in earlier periods.
3.  **Window Overlaps / Definition Changes**: If the data aggregation window (e.g., daily, weekly) or the definition of key metrics has changed over time, it can introduce inconsistencies. Overlaps might occur if data sources are combined without proper de-duplication or if reporting periods are not clearly aligned. Such issues can make time-series analysis challenging and potentially misleading.
4.  **Implicit vs. Explicit Signals**: The data often captures explicit actions (clicks, impressions) but may miss implicit user intent or satisfaction. For example, a user who finds what they need quickly might not generate many explicit signals but was highly satisfied. This `dark matter` of user behavior is typically unobservable.
5.  **Sampling or Aggregation Effects**: If the data is sampled or pre-aggregated before it reaches this dataset, some granularity or specific edge cases might have been lost. This can limit the ability to perform very fine-grained analysis or detect subtle patterns.

In [18]:
# --- 4.1 Check for Unbalanced History (e.g., per 'client_hash_id' or 'content_hash_id') ---
# For 'fact_content_daily_performance', we might look at content_hash_id or client_hash_id to check history length.

print("\n--- Date range per 'content_hash_id' (top 5 examples) ---")
if 'content_hash_id' in df.columns and 'report_date' in df.columns:
    # Ensure report_date is datetime for proper calculations
    df['report_date'] = pd.to_datetime(df['report_date'], errors='coerce')

    content_date_ranges = df.groupby('content_hash_id')['report_date'].agg(['min', 'max'])
    display(content_date_ranges.head())

    # Distribution of history length
    content_date_ranges['history_length_days'] = (content_date_ranges['max'] - content_date_ranges['min']).dt.days
    print("\nDistribution of content history length (in days):")
    display(content_date_ranges['history_length_days'].describe())
else:
    print("Warning: 'content_hash_id' or 'report_date' column not found to check content history.")

# --- 4.2 Check for potential Source-Specific Biases over Time ---
# Using 'gsc_data_available' to see how the presence of GSC data changes over time.

print("\n--- Distribution of 'gsc_data_available' over 'report_date' ---")
if 'gsc_data_available' in df.columns and 'report_date' in df.columns:
    # Group by report_date and gsc_data_available, then count occurrences
    gsc_availability_over_time = df.groupby(['report_date', 'gsc_data_available']).size().unstack(fill_value=0)
    display(gsc_availability_over_time)
    print("This shows the count of rows per day where GSC data was available (True) or not (False).")
else:
    print("Note: 'gsc_data_available' or 'report_date' column not found to analyze source-specific biases over time.")

# --- 4.3 Check for explicit Window Overlaps (if applicable) ---
print("\n--- Checking for potential window overlaps / exact duplicates based on all columns ---")
initial_rows = len(df)
cleaned_df = df.drop_duplicates()
if initial_rows > len(cleaned_df):
    print(f"Found {initial_rows - len(cleaned_df)} exact duplicate rows. This might indicate unintended window overlaps or data ingestion issues.")
else:
    print("No exact duplicate rows found, suggesting no obvious full row window overlaps.")


--- Date range per 'content_hash_id' (top 5 examples) ---


,min,max
content_hash_id,,
content_01d48890af12916f,2025-01-27,2025-01-27
content_0208673d5d06fe3f,2025-01-27,2025-01-27
content_0217be03126aa7a5,2025-01-27,2025-01-28
content_02a606212c01ce89,2025-01-27,2025-01-28
content_04bc718f0bccb54e,2025-01-27,2025-01-28



Distribution of content history length (in days):


,history_length_days
count,336.000000
mean,0.488095
std,0.500604
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,1.000000



--- Distribution of 'gsc_data_available' over 'report_date' ---


gsc_data_available,True
report_date,
2025-01-27,303
2025-01-28,197


This shows the count of rows per day where GSC data was available (True) or not (False).

--- Checking for potential window overlaps / exact duplicates based on all columns ---
No exact duplicate rows found, suggesting no obvious full row window overlaps.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.